# ISLES'24 parallel persistent downloader

Run **Runtime → Run all**. This notebook downloads the ~99 GB `train.7z` archive as independent HTTP byte-range chunks in parallel and stores every completed chunk in Google Drive.

If Colab disconnects, reopen this same notebook and run all again. Finished chunks are skipped and partial chunks resume. If the older sequential downloader already produced a partial `train.7z`, this workflow preserves it as the already-downloaded prefix instead of throwing it away.

Default: **4 parallel workers**, **1 GiB per chunk**. This is intentionally conservative to avoid Zenodo throttling and Google Drive FUSE instability.


## 1. Mount Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 2. Clone latest repository


In [ ]:
ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PARTS = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z.parts')
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python', str(SCRIPT),
    '--archive', str(ARCHIVE),
    '--parts-dir', str(PARTS),
    '--workers','4',
    '--part-size-gib','1',
    '--retries','20',
], check=True)


## 3. Parallel chunk download to persistent Drive


In [ ]:
ARCHIVE = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z')
PARTS = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024/raw/train.7z.parts')
ARCHIVE.parent.mkdir(parents=True, exist_ok=True)

subprocess.run([
    'python','scripts/download_isles2024_parallel.py',
    '--archive', str(ARCHIVE),
    '--parts-dir', str(PARTS),
    '--workers','4',
    '--part-size-gib','1',
    '--retries','20',
], check=True)


## 4. Verify persistent result


In [ ]:
print('Archive:', ARCHIVE)
print('Exists:', ARCHIVE.exists())
if ARCHIVE.exists():
    print('Size GiB:', round(ARCHIVE.stat().st_size/(1024**3), 2))
print('Chunks directory exists:', PARTS.exists())
print('Successful completion means the official Zenodo MD5 was verified.')


## Reconnection rule

If Colab disconnects, simply run the notebook again from the top. Do **not** delete the `.parts` directory or `.prefix` file. They are the persistent resume state.
